# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

c:\Users\aidan\miniconda3\envs\cse446\python.exe


In [6]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

import sys
!{sys.executable} -m pip install numpy==1.21.0


  Using cached numpy-1.21.0-cp39-cp39-win_amd64.whl.metadata (2.0 kB)
Using cached numpy-1.21.0-cp39-cp39-win_amd64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-model-optimization 0.8.0 requires numpy~=1.23, but you have numpy 1.21.0 which is incompatible.


In [9]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

import sys
!{sys.executable} -m pip install tensorflow==2.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cse446 0.0.1 requires numpy==1.21.0, but you have numpy 2.0.2 which is incompatible.
scipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.0.2 which is incompatible.
tensorflow-model-optimization 0.8.0 requires numpy~=1.23, but you have numpy 2.0.2 which is incompatible.


  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
   ---------------------------------------- 0.0/284.1 MB ? eta -:--:--
    --------------------------------------- 6.3/284.1 MB 32.2 MB/s eta 0:00:09
   - -------------------------------------- 12.3/284.1 MB 32.2 MB/s eta 0:00:09
   -- ------------------------------------- 19.1/284.1 MB 31.8 MB/s eta 0:00:09
   --- ------------------------------------ 25.2/284.1 MB 30.1 MB/s eta 0:00:09
   ---- ----------------------------------- 29.9/284.1 MB 29.2 MB/s eta 0:00:09
   ----- ---------------------------------- 37.2

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.0
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [11]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [12]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.


X = df.drop(columns=["Class"])
y = df["Class"]


In [13]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)



In [16]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [17]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes = num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes = num_classes)




In [22]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

wine_model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])



In [30]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

wine_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy', 
    metrics = ['accuracy']
)

trained_model = wine_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs = 20,
    batch_size = 8,
    validation_split=0.2
)


Epoch 1/20
13/13 [==============================] - 1s 11ms/step - loss: 0.9304 - accuracy: 0.6869 - val_loss: 0.6513 - val_accuracy: 0.9600
Epoch 2/20
13/13 [==============================] - 0s 3ms/step - loss: 0.6150 - accuracy: 0.9192 - val_loss: 0.4289 - val_accuracy: 1.0000
Epoch 3/20
13/13 [==============================] - 0s 3ms/step - loss: 0.4075 - accuracy: 0.9293 - val_loss: 0.2807 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 3ms/step - loss: 0.2788 - accuracy: 0.9596 - val_loss: 0.1958 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 3ms/step - loss: 0.1946 - accuracy: 0.9697 - val_loss: 0.1468 - val_accuracy: 1.0000
Epoch 6/20
13/13 [==============================] - 0s 3ms/step - loss: 0.1412 - accuracy: 0.9798 - val_loss: 0.1198 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 3ms/step - loss: 0.1091 - accuracy: 0.9798 - val_loss: 0.1014 - val_accuracy: 1.0000
Epoch 8/20
13/13 [=

In [35]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

loss, accuracy = wine_model.evaluate(X_test_scaled, y_test_cat)
print(f"Test Accuracy: {accuracy:.4f}\n")

raw_predictions = wine_model.predict(
    X_test_scaled, 
    batch_size = 8, 
    verbose="auto",
    steps=None,
    callbacks=None
)

y_pred = np.argmax(raw_predictions, axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


2/2 [==============================] - 0s 2ms/step - loss: 0.0286 - accuracy: 1.0000
Test Accuracy: 1.0000

7/7 [==============================] - 0s 840us/step
Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54



In [37]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes
import os
converter = tf.lite.TFLiteConverter.from_keras_model(wine_model)

tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_base.tflite") / 1024
print(f"Base TFLite model size: {file_size_kb:.2f} KB")



INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmp4nh6i6cu\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmp4nh6i6cu\assets


Base TFLite model size: 22.59 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [54]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

        pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

        pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        pass

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    model_tflite = converter.convert()

    with open(filename, "wb") as f:
        f.write(model_tflite)   


    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    y_pred_tf = []

    # Loop through each class in the model
    for i in range (len(X_test)):
        # take a sample so the model can read
        test_sample = np.expand_dims(X_test[i], axis =0).astype(np.float32)

        # checking to see if the input is qunatized
        if input_scale !=0.0:
            # add the scalar to the 0 point,  and change type
            test_sample = test_sample / input_scale + input_zero_point
            test_sample = test_sample.astype(input_details["dtype"])

        # upload the data into the model    
        interpreter.set_tensor(input_details["index"], test_sample)

        # run model prediction
        interpreter.invoke()

        # get the raw preidciton from the model.
        output_data = interpreter.get_tensor(output_details["index"])

        if output_scale != 0.0:
            output_data = output_data.astype(np.float32)
            output_data = (output_data - output_zero_point) * output_scale

        prediction = np.argmax(output_data)
        y_pred_tf.append(prediction)

    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {os.path.getsize(filename) / 1024:.2f} KB")
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_tf))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_tf))



In [52]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# Test 1: Dynamic Quantization
print("--- Dynamic Quantization ---")
quantize_and_evaluate(
    model=wine_model, 
    X_test=X_test_scaled,       # Make sure to pass the scaled test data!
    y_test_cat=y_test_cat, 
    quant_type='dynamic', 
    filename='model_dynamic.tflite'
)

# Test 2: Float16 Quantization
print("\n--- Float16 Quantization ---")
quantize_and_evaluate(
    model=wine_model, 
    X_test=X_test_scaled, 
    y_test_cat=y_test_cat, 
    quant_type='float16', 
    filename='model_float16.tflite'
)

# Test 3: Int8 Quantization
print("\n--- Int8 Quantization ---")
quantize_and_evaluate(
    model=wine_model, 
    X_test=X_test_scaled, 
    y_test_cat=y_test_cat, 
    quant_type='int8', 
    filename='model_int8.tflite'
)


--- Dynamic Quantization ---
INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpsds28nry\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpsds28nry\assets



DYNAMIC TFLite model size: 10.69 KB
Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


--- Float16 Quantization ---
INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmp_g0vnuwi\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmp_g0vnuwi\assets



FLOAT16 TFLite model size: 13.21 KB
Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


--- Int8 Quantization ---
INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpja04sqk9\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpja04sqk9\assets



INT8 TFLite model size: 7.98 KB
Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54



c:\Users\aidan\miniconda3\envs\cse446\lib\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


## Problem 1 - Part (c)

### Pruning

In [55]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch _size * epochs)


dataset_size = len(X_train_scaled)
batch_size = 8
epochs = 20
total_steps = int(dataset_size / batch_size) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=total_steps
)




In [58]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude  

wine_model_pruning = tf.keras.Sequential([

    prune_low_magnitude(
        tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)), 
        pruning_schedule=pruning_schedule
    ),
    
    prune_low_magnitude(
        tf.keras.layers.Dense(32, activation='relu'), 
        pruning_schedule=pruning_schedule
    ),
    
    prune_low_magnitude(
        tf.keras.layers.Dense(3, activation='softmax'), 
        pruning_schedule=pruning_schedule
    )
])


In [60]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

wine_model_pruning.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy', 
    metrics = ['accuracy']
)

trained_model_pruning = wine_model_pruning.fit(
    X_train_scaled,
    y_train_cat,
    epochs = 10,
    batch_size = batch_size,
    validation_split=0.2,
    callbacks =[tfmot.sparsity.keras.UpdatePruningStep()]
)

    

Epoch 1/10
13/13 [==============================] - 2s 22ms/step - loss: 1.2265 - accuracy: 0.2525 - val_loss: 1.0102 - val_accuracy: 0.4800
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 0.8498 - accuracy: 0.7576 - val_loss: 0.7064 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.6019 - accuracy: 0.9697 - val_loss: 0.5049 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 5ms/step - loss: 0.4214 - accuracy: 0.9899 - val_loss: 0.3731 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 5ms/step - loss: 0.2933 - accuracy: 0.9899 - val_loss: 0.2752 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 7ms/step - loss: 0.2097 - accuracy: 0.9899 - val_loss: 0.2111 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.1534 - accuracy: 0.9798 - val_loss: 0.1713 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [62]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# Remove the priuning warppers 
model_for_export = tfmot.sparsity.keras.strip_pruning(wine_model_pruning)

# Covert to TFlite
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
tflite_pruned_model = converter.convert()

# Save the model to drive
with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

# print the file size 
file_size_kb = os.path.getsize("model_pruned.tflite") / 1024
print(f"Pruned TFLite model size: {file_size_kb:.2f} KB")




INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpqp9sgy0p\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpqp9sgy0p\assets


Pruned TFLite model size: 14.14 KB


In [63]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_prob = model_for_export.predict(X_test_scaled)

y_pred_prune = np.argmax(y_pred_prob, axis = 1)



print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_prune))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_prune))


2/2 [==============================] - 0s 4ms/step
Confusion Matrix:
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54



## Problem 1 - Part (d)

### Knowledge Distillation

In [71]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = tf.keras.Sequential([


        tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),

        tf.keras.layers.Dense(32, activation='relu'), 

        tf.keras.layers.Dense(3, activation='softmax'), 

])

In [73]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_soft_labels = wine_model.predict(X_train_scaled)

4/4 [==============================] - 0s 2ms/step


In [74]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# concatenate together along axis 1 
y_train_combined = np.concatenate([y_train_cat, teacher_soft_labels], axis=1)

def distillation_loss(y_true_combined, y_pred):

    # get the categories out again by slicing and seperating down the middle 
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    # get losses via cat cross entropy 
    loss_hard = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    loss_soft = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    alpha = 0.5
    total_loss = (alpha * loss_hard) + ((1 - alpha) * loss_soft)
    return total_loss

In [75]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer = 'adam',
    loss = distillation_loss, 
    metrics = ['accuracy']
)

trained_student = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs = 10,
    batch_size = 8,
    validation_split=0.2
)


Epoch 1/10
13/13 [==============================] - 1s 14ms/step - loss: 0.9724 - accuracy: 0.6061 - val_loss: 0.8254 - val_accuracy: 0.8400
Epoch 2/10
13/13 [==============================] - 0s 3ms/step - loss: 0.7048 - accuracy: 0.9394 - val_loss: 0.6103 - val_accuracy: 0.9200
Epoch 3/10
13/13 [==============================] - 0s 3ms/step - loss: 0.5180 - accuracy: 0.9596 - val_loss: 0.4456 - val_accuracy: 1.0000
Epoch 4/10
13/13 [==============================] - 0s 3ms/step - loss: 0.3733 - accuracy: 0.9798 - val_loss: 0.3279 - val_accuracy: 1.0000
Epoch 5/10
13/13 [==============================] - 0s 3ms/step - loss: 0.2706 - accuracy: 0.9798 - val_loss: 0.2472 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 3ms/step - loss: 0.1947 - accuracy: 0.9899 - val_loss: 0.1930 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 3ms/step - loss: 0.1478 - accuracy: 0.9899 - val_loss: 0.1560 - val_accuracy: 1.0000
Epoch 8/10
13/13 [=

In [76]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_student_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_student_model)

file_size_kb = os.path.getsize("model_kd.tflite") / 1024
print(f"KD TFLite model size: {file_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpwfkamy95\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpwfkamy95\assets


KD TFLite model size: 14.14 KB


In [77]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_probs = student_model.predict(X_test_scaled)
y_pred_student = np.argmax(y_pred_probs, axis=1)

y_true = np.argmax(y_test_cat, axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_student))

print("\nClassification Report:")
print(classification_report(y_true, y_pred_student))

2/2 [==============================] - 0s 3ms/step
Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54



## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [81]:
# ==========================================
# PRUNING STUDENT MODEL 

# ==========================================

# Setup the pruning schedule for 10 epochs of fine-tuning
dataset_size = len(X_train_scaled)
batch_size = 8
epochs = 10
total_steps = int(dataset_size / batch_size) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=total_steps
)

# Place the student model in the pruning wrapper
pruned_student_model = tfmot.sparsity.keras.prune_low_magnitude(
    student_model, 
    pruning_schedule=pruning_schedule
)

# compile with the standard compiler
pruned_student_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

# pass in the data and 
pruned_student_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=0 # dont print training data
)

# remove the wrappers - and zeros 
stripped_student_model = tfmot.sparsity.keras.strip_pruning(pruned_student_model)


# ==========================================
# QUANTIZE TO INT8 
# ==========================================

# convert from keras to TFLite for quantization
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_student_model)

# copy the quanitazation rules 
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# voert
tflite_student_model = converter.convert()
student_filename = "model_student_pipeline.tflite"

with open(student_filename, "wb") as f:
    f.write(tflite_student_model)



# Print the final combined file size
file_size_kb = os.path.getsize(student_filename) / 1024
print(f"Final Student TFLite model size: {file_size_kb:.2f} KB")

# Load the interpreter
interpreter = tf.lite.Interpreter(model_path=student_filename)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]

y_pred_tf = []

# Loop through the test set
for i in range(len(X_test_scaled)):
    test_sample = np.expand_dims(X_test_scaled[i], axis=0).astype(np.float32)
    
    if input_scale != 0.0:
        test_sample = test_sample / input_scale + input_zero_point
        test_sample = test_sample.astype(input_details["dtype"])
        
    interpreter.set_tensor(input_details["index"], test_sample)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details["index"])
    
    if output_scale != 0.0:
        output_data = output_data.astype(np.float32)
        output_data = (output_data - output_zero_point) * output_scale
        
    prediction = np.argmax(output_data)
    y_pred_tf.append(prediction)

y_true = np.argmax(y_test_cat, axis=1)

# Print the final accuracy
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_tf))

print("\nClassification Report:")
print(classification_report(y_true, y_pred_tf))

INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpov0c_mws\assets


INFO:tensorflow:Assets written to: C:\Users\aidan\AppData\Local\Temp\tmpov0c_mws\assets
c:\Users\aidan\miniconda3\envs\cse446\lib\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Final Student TFLite model size: 5.82 KB

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54



# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
